# Hypothesis Engine -- Training with GRPO (TRL + Unsloth)

This notebook trains a small LLM (Qwen2.5-1.5B) to become a better scientist
using Group Relative Policy Optimization (GRPO) against the Hypothesis Engine
environment.

**Requirements:**
- Google Colab with T4 GPU (free tier works!)
- ~15 minutes for 50 training episodes

**What this demonstrates:**
- Real RL training loop with verifiable reward curves
- Reward improvement over training episodes
- LLM learns to experiment, hypothesize, and predict

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth
!pip install -q "trl>=0.12.0" transformers accelerate bitsandbytes datasets
!pip install -q numpy rich openenv-core==0.2.1
!pip install -q git+https://github.com/AbhinavDubey30/OpenMax.git

In [ ]:
# Step 2: Load model with Unsloth (4-bit quantized for Colab)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

# Add LoRA adapters for efficient training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: Qwen2.5-1.5B-Instruct (4-bit + LoRA)")

In [ ]:
# Step 3: Define Hypothesis Engine reward function for GRPO
import json, re, random
from hypothesis_engine import HypothesisEngine

SYSTEM_PROMPT = """You are a scientist investigating a black-box system.
You must discover the hidden mathematical rule by running experiments,
forming hypotheses, and making predictions.

Respond with EXACTLY ONE JSON action per turn. Valid actions:
- {"action": "experiment", "inputs": {"x": VALUE}}
- {"action": "hypothesize", "expression": "MATH_EXPRESSION"}
- {"action": "predict", "predictions": [v1, v2, ...]}
- {"action": "get_status"}

Strategy:
1. Run 5-10 experiments with diverse x values
2. Look for patterns in input-output pairs
3. Form a hypothesis (e.g., "2*x + 3")
4. Submit predictions for all test cases
"""

def extract_json_action(text):
    """Extract a JSON action from LLM output."""
    patterns = [
        r'```json\s*({.*?})\s*```',
        r'```\s*({.*?})\s*```',
        r'({\s*"action"\s*:.*?})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                continue
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return None


def hypothesis_reward_fn(completions, prompts, **kwargs):
    """GRPO reward function: score completions based on action quality.
    
    Rewards:
    - Valid experiment with good inputs: +0.5
    - Valid hypothesis with expression: +1.0
    - Valid predictions: +1.5
    - Malformed action: -1.0
    - JSON parse failure: -1.0
    """
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        try:
            action = extract_json_action(text)
            if action is None:
                rewards.append(-1.0)
                continue
            action_type = action.get("action", "").lower()
            if action_type == "experiment":
                inputs = action.get("inputs", {})
                if inputs and all(isinstance(v, (int, float)) for v in inputs.values()):
                    rewards.append(0.5)
                else:
                    rewards.append(-0.5)
            elif action_type == "hypothesize":
                expr = action.get("expression", "")
                if expr and len(expr) > 0:
                    rewards.append(1.0)
                else:
                    rewards.append(-0.5)
            elif action_type == "predict":
                preds = action.get("predictions", [])
                if isinstance(preds, list) and len(preds) > 0:
                    rewards.append(1.5)
                else:
                    rewards.append(-0.5)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(-1.0)
    return rewards

print("Reward function defined.")

In [ ]:
# Step 4: Generate training prompts from the environment
from datasets import Dataset

def generate_training_prompts(n_prompts=100):
    """Generate diverse prompts from Hypothesis Engine worlds."""
    prompts = []
    for i in range(n_prompts):
        difficulty = random.choice([1, 2, 3])
        env = HypothesisEngine(difficulty=difficulty, experiment_budget=15)
        obs = env.reset()
        world = obs.get("world", {})
        prompt = (
            f"You are investigating a black-box system.\n"
            f"World: {world.get('world_name', '?')}\n"
            f"Description: {world.get('description', '')}\n"
            f"Variables: {world.get('variables', [])}\n"
            f"Budget: {obs.get('experiments_remaining', 0)} experiments\n"
            f"You have {len(obs.get('test_cases', []))} test cases to predict.\n\n"
            f"Plan your investigation. Start by running experiments.\n"
            f"Respond with a JSON action."
        )
        prompts.append({"prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]})
    return prompts

training_prompts = generate_training_prompts(100)
train_dataset = Dataset.from_list(training_prompts)
print(f"Generated {len(training_prompts)} training prompts from Hypothesis Engine")

In [ ]:
# Step 5: GRPO Training with TRL
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./hypothesis_engine_grpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_completion_length=256,
    num_generations=4,
    logging_steps=5,
    save_steps=50,
    warmup_steps=10,
    bf16=True,
    report_to="none",
)

FastLanguageModel.for_training(model)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=hypothesis_reward_fn,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 6: Evaluate and visualize results
import matplotlib.pyplot as plt

def run_eval_episode(model, tokenizer, difficulty=1, max_steps=12):
    """Run one eval episode and return final reward."""
    env = HypothesisEngine(difficulty=difficulty, experiment_budget=max_steps)
    obs = env.reset()
    world = obs.get("world", {})
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"World: {world.get('world_name','?')}\nDescription: {world.get('description','')}\nVariables: {world.get('variables',[])}\nBudget: {obs.get('experiments_remaining',0)}\nBegin!"},
    ]
    
    done = False
    step = 0
    total_reward = 0.0
    
    FastLanguageModel.for_inference(model)
    while not done and step < max_steps + 5:
        input_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(input_ids=input_ids, max_new_tokens=256, temperature=0.7, do_sample=True)
        response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
        messages.append({"role": "assistant", "content": response})
        
        action = extract_json_action(response)
        if action is None:
            action = {"action": "experiment", "inputs": {"x": float(step)}}
        
        obs, reward, done, info = env.step(action)
        total_reward += reward
        step += 1
        
        if done:
            return info.get("final_reward", {}).get("total_reward", total_reward)
        messages.append({"role": "user", "content": obs.get("message", "")[:200]})
    
    return total_reward

# Run evaluation
print("Evaluating trained model on 10 episodes...")
rewards = []
for i in range(10):
    r = run_eval_episode(model, tokenizer, difficulty=1)
    rewards.append(r)
    print(f"  Episode {i+1}: {r:.1f}/100")

avg_reward = sum(rewards) / len(rewards)
print(f"\nAverage trained reward: {avg_reward:.1f}/100")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 11), rewards, color="#4c6ef5", alpha=0.8)
ax.axhline(y=avg_reward, color="red", linestyle="--", label=f"Mean: {avg_reward:.1f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward (0-100)")
ax.set_title("Hypothesis Engine: Post-Training Evaluation")
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Results saved to training_results.png")

In [ ]:
# Step 7: Save the trained model
model.save_pretrained("hypothesis_scientist_lora")
tokenizer.save_pretrained("hypothesis_scientist_lora")
print("Trained model saved to hypothesis_scientist_lora/")
print("\n=== Training Summary ===")
print(f"  Model: Qwen2.5-1.5B-Instruct + LoRA (4-bit)")
print(f"  Method: GRPO (Group Relative Policy Optimization)")
print(f"  Environment: Hypothesis Engine v2.0")
print(f"  Training prompts: {len(training_prompts)}")
print(f"  Average eval reward: {avg_reward:.1f}/100")